# PCam fine-tune on Colab T4

Fine-tune a pretrained ResNet-18 on the Kaggle Histopathologic Cancer Detection
(PCam) dataset. Target val AUC ≈ 0.94–0.96 in 2 epochs.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Also:** accept the competition rules once at
<https://www.kaggle.com/c/histopathologic-cancer-detection/rules>,
otherwise the Kaggle API refuses to download the data.

## Cell 1 — GPU check + Kaggle auth

In [ ]:
!nvidia-smi
from google.colab import files
print("kaggle.json dosyanı yükle:")
files.upload()

!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install timm

## Cell 2 — Download dataset (~7 GB)

In [ ]:
!kaggle competitions download -c histopathologic-cancer-detection
!unzip -q histopathologic-cancer-detection.zip -d pcam
!ls pcam | head

## Cell 3 — Data pipeline

In [ ]:
import torch, pandas as pd, numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA = Path('pcam'); TRAIN = DATA / 'train'
device = torch.device('cuda')

df = pd.read_csv(DATA / 'train_labels.csv')
train_df, val_df = train_test_split(df, test_size=0.1, stratify=df.label, random_state=42)
print(f"train={len(train_df)}, val={len(val_df)}, pozitif oran={df.label.mean():.3f}")

IM_MEAN, IM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
    T.ColorJitter(0.1, 0.1, 0.1), T.ToTensor(), T.Normalize(IM_MEAN, IM_STD),
])
val_tf = T.Compose([T.ToTensor(), T.Normalize(IM_MEAN, IM_STD)])

class PCam(Dataset):
    def __init__(self, df, tf): self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(TRAIN / f'{r.id}.tif').convert('RGB')
        return self.tf(img), torch.tensor(r.label, dtype=torch.float32)

train_loader = DataLoader(PCam(train_df, train_tf), batch_size=256,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(PCam(val_df, val_tf), batch_size=512,
                          shuffle=False, num_workers=2, pin_memory=True)

## Cell 4 — Model + training

In [ ]:
import timm, torch.nn as nn
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

model = timm.create_model('resnet18', pretrained=True, num_classes=1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
crit = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler()

EPOCHS = 2
for ep in range(EPOCHS):
    model.train(); total = 0
    for x, y in tqdm(train_loader, desc=f'epoch {ep+1} train'):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x).squeeze(-1)
            loss = crit(logits, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item()

    model.eval(); preds, labs = [], []
    with torch.no_grad(), torch.cuda.amp.autocast():
        for x, y in tqdm(val_loader, desc=f'epoch {ep+1} val'):
            logits = model(x.to(device)).squeeze(-1)
            preds.extend(torch.sigmoid(logits).float().cpu().numpy())
            labs.extend(y.numpy())
    print(f'epoch {ep+1}: train_loss={total/len(train_loader):.4f}, val_AUC={roc_auc_score(labs, preds):.4f}')

torch.save(model.state_dict(), 'resnet18_pcam.pth')
files.download('resnet18_pcam.pth')